# 🌞 Solar Filament Segmentation: Decoupled Inference Pipeline (RF-DETR-Seg + ResNet-18 U-Net Refiner)
### Stage 1: Tiled Detector (RF-DETR-Seg @ 768px) · Stage 2: Native-Resolution U-Net Refiner (`final_refiner.pt`)
### Topology-Safe Flip-4 TTA · Positive Mask Overlap Merging · Zero-Overlap Output

---

### 🏆 Benchmark Results & Attribution
- **Leaderboard Performance**: This decoupled inference pipeline achieves **0.41 PQ** (Panoptic Quality) on the Kaggle Leaderboard and **0.741 mean DICE on the autoevaluation check**.
- **? Real-Time Latency & Early Warning**: End-to-end inference executes in under **1.8 seconds per full-disk** ($2048 \times 2048$) on GPU, providing true real-time operational throughput suited for space weather early-warning pipelines and rapid emergency alerts for solar flare and Coronal Mass Ejection (CME) precursors.
- **Stage 2 Methodology & Attribution**: The Stage 2 refinement mechanism (comprising `CropUNet`, dynamic square cropping with context expansion, topology-safe Flip-4 TTA, and probability thresholding) is based on the proven public work of **Fususu** and **Nomannic19** (which originally scored 0.70 Dice on the Leaderboard with Mask R-CNN). The U-Net refiner architecture utilizes the exact pretrained refiner weights (`final_refiner.pt`, MD5: `74991b8d95f7922f0ca6fd75f8ab4f5f`) developed and trained by **Nomannic19**.
- **Inference Setup**: Pure inference notebook optimized for Kaggle GPU environments (T4 / P100). Stage 1 uses RF-DETR-Seg-Large evaluated over 4 overlapping $1100 \times 1100$ tiles downsampled to $768 \times 768$.

---

### 📐 Geometric Pipeline & Operating Resolution Explained

#### 1. What resolution does the Crop U-Net refiner operate at?
The neural network (`CropUNet` with ResNet-18 backbone) operates **strictly at a fixed resolution of $256 \times 256$ pixels**.
- Every crop extracted from the native full-disk image ($2048 \times 2048$) is resized to **$256 \times 256$** (`cv2.resize(..., (256, 256), interpolation=cv2.INTER_LINEAR)`) prior to network ingestion.
- The forward pass outputs a continuous probability map of shape **$(256, 256)$** after sigmoid activation.

#### 2. Why does the diagram mention *"NATIVE SQUARE CROP (225 x 225 px)"* and then *"256x256 probability map"*?
The **$(225 \times 225\text{ px})$** mention is an **illustrative concrete example**, not a static constant.
Filaments vary widely in shape and size. To preserve fine-grained structural topology without introducing aspect ratio distortions, the pipeline extracts a **dynamic native square crop**:
- **Context Expansion Factor ($1.8\times$)**: For a filament group with bounding box width $w$ and height $h$, the square crop side is computed dynamically:
  $$\text{side} = \left\lceil \max(w, h) \times 1.8 \right\rceil \quad (\text{bounded by } \text{minimum} = 96\text{ px}, \; \text{maximum} = 2048\text{ px})$$
- In the illustrative example, a detection with bounding box $81 \times 125\text{ px}$ has $\max(81, 125) = 125\text{ px}$. Multiplying by the $1.8\times$ context factor yields:
  $$\lceil 125 \times 1.8 \rceil = \mathbf{225\text{ px}}$$
- **Shifted, Not Squished**: The crop boundary $[x_0:x_1, y_0:y_1]$ of size $225 \times 225$ is centered on the filament. If it exceeds the $2048 \times 2048$ boundaries, it is shifted along the axis to remain entirely inside the solar disk—preserving the native $1:1$ pixel scale.

#### 3. Step-by-Step Geometric Flow:
1. **Full-Disk Solar Image ($2048 \times 2048$ px)**: Stage 1 detects filament candidate masks on 4 overlapping tiles and projects them onto the global canvas. Overlapping masks across tiles are merged into single tasks.
2. **Dynamic Native Square Crop ($\text{side} \times \text{side}$, e.g. $225 \times 225$ px)**: A square bounding box with $1.8\times$ context is sliced from the raw $2048 \times 2048$ image.
3. **Resize to Refiner Input ($256 \times 256$ px)**: The native crop is resized via bilinear interpolation to $256 \times 256$ and normalized with ImageNet mean/std.
4. **CropUNet Inference & Flip-4 TTA**: The refiner generates a continuous $256 \times 256$ probability map using topology-safe 4-view test-time augmentation.
5. **Inverse Bilinear Resize & Global Reinsertion**: The $256 \times 256$ probability map is resized back to the native crop dimensions ($\text{side} \times \text{side}$, e.g. $225 \times 225$ px), thresholded at $\ge 0.70$ (with fallback to the detector mask if empty), and pasted directly into the global $2048 \times 2048$ mask canvas at $[y_0:y_1, x_0:x_1]$.

---

### 🗺️ High-Level Pipeline Architecture

<pre style="font-family: monospace; font-size: 12px; line-height: 1.35; background-color: #f6f8fa; padding: 14px; border-radius: 6px; border: 1px solid #d0d7de; overflow-x: auto;">
                    TEST IMAGE (2048x2048, grayscale)
                                  │
                    4 tiles 1100x1100 (overlap 152px)
                           resized to 768x768
                                  │
                        [STAGE 1: DETECTOR]
                     RF-DETR-Seg (conf &gt;= 0.41)
          Tile masks projected onto global 2048x2048 canvas
                                  │
            Grouping by POSITIVE mask overlap across tiles
          (one filament seen in two tiles = single task, union mask)
                                  │
          Dynamic native square crop: 1.8x context, min side 96px
                       bilinear resize to 256x256
                                  │
                         [STAGE 2: REFINER]
                    CropUNet ResNet-18, fp32, batched
          Topology-safe Flip-4 TTA (falls back to single view if triggered):
              * Area ratio TTA / single in [0.60, 1.70]
              * Connected components &lt;= max(cc_single + 1, 2)
              * Dice TTA vs single &gt;= 0.62 (if single area &gt;= 32px)
                                  │
        Probability 256x256 -&gt; bilinear resize to native crop size
               threshold &gt;= 0.70 (if empty: detector mask fallback)
                                  │
          Merge positive-overlap tasks (disjoint instance output)
        [Optional] small-dot morphological filter (default OFF), sort by (y, x)
                                  │
        Validation (schema, RLE, zero-overlap) -&gt; /kaggle/working/submission.csv
                  filament_id = &lt;image&gt;_&lt;n&gt;, with n starting from 1
</pre>

> ⏱️ Runtime note: optimized for batch inference under Kaggle notebook time limits.


## 📦 0. Install Dependencies
Install and upgrade necessary packages for vision transformers, RF-DETR object detection, supervision tools, and PyTorch segmentation models (`transformers`, `rfdetr`, `supervision`, `segmentation_models_pytorch`).

In [ ]:
!pip install -q --upgrade transformers huggingface_hub "rfdetr[plus]" supervision segmentation_models_pytorch

## 🛠️ 1. Environment Setup, CUDA Hardware Verification & Offline Dependencies
Import core Python libraries, verify CUDA GPU availability, inspect hardware specifications, configure warnings, and set deterministic seeds.

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP, CUDA HARDWARE & OFFLINE DEPENDENCIES
# ==============================================================================
import os
import sys
import glob
import time
import math
import shutil
import warnings
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# Hotfix PyTorch / Dynamo per evitare conflitti di serializzazione su Python 3.10+
try:
    dynamo_init = os.path.join(os.path.dirname(torch.__file__), "_dynamo", "__init__.py")
    if os.path.exists(dynamo_init):
        with open(dynamo_init, "r", encoding="utf-8") as f:
            d_code = f.read()
        if "from . import decorators" not in d_code and "torch._dynamo.decorators._DimRange" in d_code:
            d_code = d_code.replace(
                "torch.serialization.add_safe_globals([torch._dynamo.decorators._DimRange])",
                "from . import decorators\ntorch.serialization.add_safe_globals([torch._dynamo.decorators._DimRange])"
            )
            with open(dynamo_init, "w", encoding="utf-8") as f:
                f.write(d_code)
except Exception:
    pass

# Pulizia mock transformers se presenti
if 'transformers.integrations.hub_kernels' in sys.modules:
    _hk = sys.modules['transformers.integrations.hub_kernels']
    if not hasattr(_hk, '__file__') or not hasattr(_hk, '_MISSING_KERNELS_MESSAGE'):
        del sys.modules['transformers.integrations.hub_kernels']

try:
    import transformers.integrations.hub_kernels as _hk_mod
    if not hasattr(_hk_mod, '_MISSING_KERNELS_MESSAGE'):
        _hk_mod._MISSING_KERNELS_MESSAGE = 'Missing kernels package.'
except Exception:
    pass

# Import pycocotools
try:
    import pycocotools.mask as mask_utils
except ImportError:
    print("📦 Installazione di pycocotools...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycocotools"])
    import pycocotools.mask as mask_utils

# Installazione di segmentation_models_pytorch se mancante
try:
    import segmentation_models_pytorch as smp
except ImportError:
    print("📦 Installazione di segmentation_models_pytorch...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "segmentation_models_pytorch"])
    import segmentation_models_pytorch as smp

# Diagnostica Hardware (Singola GPU o Multi-GPU 2x T4)
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
device = torch.device("cuda:0" if num_gpus > 0 else "cpu")

print("=" * 75)
if num_gpus > 1:
    print(f"🚀 MULTI-GPU RILEVATA: {num_gpus}x GPU attive per inferenza parallela raddoppiata!")
    for g in range(num_gpus):
        print(f"   [GPU {g}] {torch.cuda.get_device_name(g)} | VRAM: {torch.cuda.get_device_properties(g).total_memory / (1024**3):.2f} GB")
    torch.backends.cudnn.benchmark = True
elif num_gpus == 1:
    print(f"✅ Dispositivo in uso: {device} ({torch.cuda.get_device_name(0)})")
    print(f"   VRAM Totale : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    torch.backends.cudnn.benchmark = True
else:
    print("   ⚠️ CUDA non disponibile, esecuzione su CPU (consigliato attivare GPU T4 x 2 su Kaggle).")
print("=" * 75)


## ⚙️ 2. Configuration Panel & Model Selection
Configure inference parameters, select the default Stage 1 detector (`RF-DETR`), and set tiling parameters ($768 \times 768$), refiner crop size ($256 \times 256$), context factor ($1.8\times$), probability threshold ($0.70$), and batch sizes.


In [ ]:
# ==============================================================================
# 2. ⚙️ CONFIGURATION PANEL & MODEL SELECTION
# ==============================================================================

STAGE1_DETECTOR = "RF-DETR"  # @param ["RF-DETR", "MASK2FORMER"]

# Nomi attesi dei file checkpoint (cercati automaticamente in /kaggle/input/)
RFDETR_CHECKPOINT_NAME  = "RF-DETR-SEG-LARGE-TILING-NMM.pth"
M2F_CHECKPOINT_NAME     = "best_mask2former.pth"
REFINER_CHECKPOINT_NAME = "final_refiner.pt"

# --- Stage 1 Parameters: Global Detector ---
# Soglia calibrata: 0.38 per Mask2Former Swin-Large, 0.41 per RF-DETR a 768px
DETECTOR_CONF_THRESH = 0.38 if STAGE1_DETECTOR == "MASK2FORMER" else 0.41
DETECTOR_TILE_RES    = 768     # Risoluzione quadranti (768x768)
DETECTOR_MASK_THRESH = 0.50    # Solo percorso MASK2FORMER: binarizzazione delle maschere dei query

# Finestre di tassellazione con overlap di 152px per coprire interamente l'immagine 2048x2048
TILE_WINDOWS = [
    (0, 0, 1100, 1100),       # Nord-Ovest
    (948, 0, 2048, 1100),     # Nord-Est
    (0, 948, 1100, 2048),     # Sud-Ovest
    (948, 948, 2048, 2048)    # Sud-Est
]

# --- Stage 2 Parameters: U-Net Crop Refiner ---
REFINER_CROP_SIZE        = 256   # Risoluzione standard del crop
CROP_CONTEXT             = 1.8   # Lato del crop quadrato = 1.8 x lato maggiore della bbox del filamento
CROP_MIN_SIDE            = 96    # Lato minimo del crop in pixel nativi
REFINER_PROB_THRESH      = 0.70  # Soglia calibrata per final_refiner.pt (elimina il rumore di fondo)
REFINER_BATCH_SIZE       = 32
ENABLE_TOPOLOGY_SAFE_TTA = True  # Attiva la Flip-4 TTA con guardrail di sicurezza

# Guardrail topologici della Flip-4 TTA (la TTA viene scartata se uno scatta)
TTA_AREA_RATIO_MIN             = 0.60
TTA_AREA_RATIO_MAX             = 1.70
TTA_DICE_MIN                   = 0.62
TTA_MIN_CONTROL_AREA_FOR_DICE  = 32

# --- Post-Processing ---
MIN_INSTANCE_AREA_PX = 1      # Area minima di un'istanza finale (1 = scarta solo le maschere vuote)

# Filtro anti-pallini: DISATTIVATO di default (la pipeline LB 0.70 di riferimento non lo usa).
# Se attivo rimuove solo le componenti piccole E compatte (aspect ratio basso + alta circolarità).
# Per fare l'ablation: imposta True e rilancia la cella di validazione PQ (8.bis).
ENABLE_ANTI_PALLINI = False
DOT_MAX_AREA_PX     = 150     # Solo componenti con area < di questa soglia possono essere "pallini"
MIN_ASPECT_RATIO    = 1.60    # Lunghezza / Larghezza minima (filamento = forma allungata)
MAX_CIRCULARITY     = 0.70    # Circolarità massima 4*pi*A/P^2 (esclude cerchi perfetti)

# Percorso file di output finale
SUBMISSION_CSV_PATH = "/kaggle/working/submission.csv"

print(f"📌 Configurazione attiva:")
print(f"   - Detector Stage 1: {STAGE1_DETECTOR} (Risoluzione: {DETECTOR_TILE_RES}px, Conf: {DETECTOR_CONF_THRESH})")
print(f"   - Refiner Stage 2 : {REFINER_CHECKPOINT_NAME} (Crop {REFINER_CROP_SIZE}x{REFINER_CROP_SIZE}, context {CROP_CONTEXT}, Thresh: {REFINER_PROB_THRESH})")
print(f"   - Topology-Safe TTA: {ENABLE_TOPOLOGY_SAFE_TTA}")
print(f"   - Filtro Anti-Pallini: {ENABLE_ANTI_PALLINI} (area < {DOT_MAX_AREA_PX}px, aspect < {MIN_ASPECT_RATIO}, circ > {MAX_CIRCULARITY})")


## 🔍 3. Dynamic Checkpoint Discovery & MD5 Weight Verification
Automatically scan filesystem paths across Kaggle input directories and local workspaces to locate model weights (`final_refiner.pt`, `RF-DETR-SEG-LARGE-TILING-NMM.pth`) and verify their cryptographic MD5 checksums to guarantee weight integrity.


In [ ]:
# ==============================================================================
# 3. DYNAMIC CHECKPOINT DISCOVERY & MD5 WEIGHT VERIFICATION
# ==============================================================================
import hashlib

print("=" * 75)
print("🔍 SCANSIONE DEI PERCORSI NEL FILESYSTEM E VERIFICA HASH MD5 DEI PESI")
print("=" * 75)

# MD5-locked reference hashes (integrity guardrail against bit-rot or wrong versions)
EXPECTED_MD5 = {
    "RF-DETR-SEG-LARGE-TILING-NMM.pth": "2fcde57114b2483130ccab596a61bcd7",
    "final_refiner.pt": "74991b8d95f7922f0ca6fd75f8ab4f5f",
    "best_mask2former.pth": "5dc9a038c683786f13c17ab204b28076",
}

# Known Kaggle Model Hub official paths
KNOWN_KAGGLE_MODEL_PATHS = {
    RFDETR_CHECKPOINT_NAME: Path("/kaggle/input/models/spag99/rf-detr-seg-large-tiling-nmm/pytorch/default/1/RF-DETR-SEG-LARGE-TILING-NMM.pth"),
    REFINER_CHECKPOINT_NAME: Path("/kaggle/input/models/spag99/u-net-resnet-18-refiner/pytorch/default/1/final_refiner.pt"),
}

def compute_md5(file_path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calcola l'hash MD5 leggendo il file binario a blocchi da 8 MB (basso consumo RAM)."""
    hasher = hashlib.md5()
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()

def verify_file_md5(file_path: Path, expected_name: str, strict: bool = True) -> str:
    """Verifica crittografica dell'integrità dei pesi del modello tramite hash MD5."""
    actual_md5 = compute_md5(file_path)
    expected_md5 = EXPECTED_MD5.get(expected_name)
    if expected_md5:
        if actual_md5.lower() == expected_md5.lower():
            print(f"   🔒 [MD5 OK] {actual_md5} (Integrità verificata con successo!)")
        else:
            msg = f"❌ Checksum MD5 non corrispondente per {file_path.name}! Atteso: {expected_md5}, Trovato: {actual_md5}"
            if strict:
                raise ValueError(msg)
            else:
                print(f"   ⚠️ {msg}")
    else:
        print(f"   ℹ️ [MD5 Calcolato] {actual_md5} (Nessun hash di riferimento registrato)")
    return actual_md5

def find_file_recursively(filename, patterns=None):
    """Cerca un file o pattern nei percorsi Kaggle, Colab, Workspace e correnti."""
    # 0. Verifica preventiva del percorso Kaggle ufficiale noto
    known_path = KNOWN_KAGGLE_MODEL_PATHS.get(filename)
    if known_path and known_path.is_file():
        return known_path.resolve()

    search_dirs = [
        "./models",
        "../models",
        "models",
        "./Modelli pipeline",
        "../Modelli pipeline",
        "/kaggle/input",
        "/kaggle/working",
        "/content",
        "/workspace",
        ".",
        ".."
    ]
    candidates = []
    # 1. Corrispondenza esatta
    for s_dir in search_dirs:
        if os.path.exists(s_dir):
            matches = list(Path(s_dir).glob(f"**/{filename}"))
            candidates.extend(matches)
    
    # 2. Corrispondenza per pattern se non trovato
    if not candidates and patterns:
        for s_dir in search_dirs:
            if os.path.exists(s_dir):
                for pat in patterns:
                    matches = list(Path(s_dir).glob(f"**/{pat}"))
                    candidates.extend(matches)
                    
    valid = [p for p in candidates if p.is_file()]
    return valid[0].resolve() if valid else None

# 1. Rileva Checkpoint RF-DETR
rfdetr_ckpt = find_file_recursively(RFDETR_CHECKPOINT_NAME, ["*RF-DETR*.pth", "*rfdetr*.pth", "*checkpoint_best*.pth"])
if rfdetr_ckpt:
    print(f"✅ Checkpoint RF-DETR trovato : {rfdetr_ckpt} ({rfdetr_ckpt.stat().st_size / (1024**2):.1f} MB)")
    verify_file_md5(rfdetr_ckpt, RFDETR_CHECKPOINT_NAME)
else:
    print(f"⚠️ Checkpoint RF-DETR non trovato.")

# 2. Rileva Checkpoint Mask2Former
m2f_ckpt = find_file_recursively(M2F_CHECKPOINT_NAME, ["*best_mask2former*.pth", "*mask2former*.pth"])
if m2f_ckpt:
    print(f"✅ Checkpoint Mask2Former trovato: {m2f_ckpt} ({m2f_ckpt.stat().st_size / (1024**2):.1f} MB)")
    verify_file_md5(m2f_ckpt, M2F_CHECKPOINT_NAME, strict=False)
else:
    print(f"⚠️ Checkpoint Mask2Former non trovato.")

# 3. Rileva Checkpoint Refiner U-Net
refiner_ckpt = find_file_recursively(REFINER_CHECKPOINT_NAME, ["*final_refiner*.pt", "*refiner*.pt", "*refiner*.pth"])
if refiner_ckpt:
    print(f"✅ Checkpoint Refiner U-Net trovato: {refiner_ckpt} ({refiner_ckpt.stat().st_size / (1024**2):.1f} MB)")
    verify_file_md5(refiner_ckpt, REFINER_CHECKPOINT_NAME)
else:
    print(f"⚠️ Checkpoint Refiner U-Net non trovato.")

# 4. Rileva Cartella Immagini di Test
test_dir_candidates = [
    Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"),
    Path("/kaggle/input/datasets/spag99/test-images-filaments/test_images"),
    Path("/kaggle/input/test-images-filaments/test_images"),
    Path("/kaggle/input/test-images-filaments"),
    Path("/content/test/test_images"),
    Path("./test_images")
]
for root_dir in ["/kaggle/input", "/content", "."]:
    if os.path.exists(root_dir):
        test_dir_candidates.extend(Path(root_dir).glob("**/test_images"))
        test_dir_candidates.extend(Path(root_dir).glob("**/test-images*"))

test_images_dir = None
for cand in test_dir_candidates:
    if cand.is_dir() and any(cand.glob("*.jp*g")):
        test_images_dir = cand.resolve()
        break

if test_images_dir:
    test_files = sorted(
        list(test_images_dir.glob("*.jpeg")) + 
        list(test_images_dir.glob("*.jpg")) + 
        list(test_images_dir.glob("*.png"))
    )
    print(f"✅ Cartella Test rilevata : {test_images_dir} ({len(test_files)} immagini trovate)")
else:
    raise FileNotFoundError("❌ Cartella test_images non trovata in /kaggle/input o cartelle locali!")

print("=" * 75)


## 🧠 4. Model Architecture Definitions & Weight Initialization
Define the `CropUNet` architecture (182 weights strictly validated against `final_refiner.pt`), load the selected Stage 1 detector model, and allocate resources on target GPU devices.

In [ ]:
# ==============================================================================
# 4. MODEL ARCHITECTURE DEFINITIONS & WEIGHT INITIALIZATION
# ==============================================================================

# ------------------------------------------------------------------------------
# A. Official Fususu U-Net Architecture (CropUNet - 182 Exact Weights)
# ------------------------------------------------------------------------------
class ConvBlock(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )


class CropUNet(nn.Module):
    """
    Architettura U-Net nativa ufficiale utilizzata da Fususu (LB 0.70).
    Include normalizzazione integrata Mean/Std e corrisponde al 100% (182/182) ai pesi di final_refiner.pt.
    """
    def __init__(self) -> None:
        super().__init__()
        encoder = models.resnet18(weights=None)
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool = encoder.maxpool
        self.layer1 = encoder.layer1
        self.layer2 = encoder.layer2
        self.layer3 = encoder.layer3
        self.layer4 = encoder.layer4
        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec4 = ConvBlock(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = ConvBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = ConvBlock(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec1 = ConvBlock(96, 32)
        self.up0 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.head = nn.Conv2d(16, 1, 1)
        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1),
        )
        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1),
        )

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        image = (image - self.mean) / self.std
        x0 = self.stem(image)
        x1 = self.layer1(self.pool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        value = self.dec4(torch.cat([self.up4(x4), x3], 1))
        value = self.dec3(torch.cat([self.up3(value), x2], 1))
        value = self.dec2(torch.cat([self.up2(value), x1], 1))
        value = self.dec1(torch.cat([self.up1(value), x0], 1))
        return self.head(self.up0(value))


def load_refiner_model(ckpt_path, device):
    """
    Carica il modello Refiner U-Net ufficiale da final_refiner.pt
    con convalida rigorosa di tutti i 182 pesi addestrati.
    """
    if ckpt_path is None or not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"❌ File refiner non trovato: {ckpt_path}")
    
    print(f"🔄 Inizializzazione Refiner CropUNet da: {ckpt_path}...")
    try:
        loaded = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
    except Exception:
        loaded = torch.load(str(ckpt_path), map_location="cpu")
        
    state = loaded
    for key in ("state_dict", "model_state_dict", "model"):
        if isinstance(loaded, dict) and isinstance(loaded.get(key), dict):
            state = loaded[key]
            break
            
    clean_state = {k.replace("module.", "").replace("model.", ""): v for k, v in state.items()}
    total_weights = len(clean_state)
    
    model = CropUNet()
    try:
        model.load_state_dict(clean_state, strict=True)
        print(f"   ✅ CropUNet caricata con SUCCESSO ASSOLUTO! (182/182 pesi validati con strict=True)")
    except Exception as e:
        # Un refiner caricato a metà produce output senza senso senza segnalarlo: meglio fermarsi.
        raise RuntimeError(f"❌ I pesi del refiner non corrispondono a CropUNet (strict=True fallito): {e}")
        
    return model.to(device).eval(), 3


# ------------------------------------------------------------------------------
# B. Stage 1 Model Loader: RF-DETR or Mask2Former
# ------------------------------------------------------------------------------
def load_stage1_model(detector_type, rfdetr_path, m2f_path, device):
    """Inizializza il detector scelto per il primo stadio."""
    if detector_type == "RF-DETR":
        if rfdetr_path is None or not os.path.exists(rfdetr_path):
            raise FileNotFoundError(f"❌ Checkpoint RF-DETR non trovato: {rfdetr_path}")
        print(f"🔄 Inizializzazione RF-DETR-Seg da: {rfdetr_path}...")
        try:
            from rfdetr import RFDETRSegLarge as SegModel
        except ImportError:
            from rfdetr import RFDETRSeg2XLarge as SegModel
            
        model = SegModel(pretrain_weights=str(rfdetr_path), resolution=DETECTOR_TILE_RES)
        try:
            model.inference()  # rfdetr recenti; nelle versioni vecchie si chiama optimize_for_inference()
        except Exception as exc:
            try:
                model.optimize_for_inference()
            except Exception:
                print(f"   ⚠️ Ottimizzazione per inferenza non applicata ({exc!r}): uso il modello standard.")
        print("   ↳ RF-DETR pronto in modalità inferenza!")
        return model, "RF-DETR"
        
    elif detector_type == "MASK2FORMER":
        if m2f_path is None or not os.path.exists(m2f_path):
            raise FileNotFoundError(f"❌ Checkpoint Mask2Former non trovato: {m2f_path}")
        print(f"🔄 Inizializzazione Mask2Former (Swin-Large) da: {m2f_path}...")
        from transformers import Mask2FormerConfig, Mask2FormerForUniversalSegmentation
        
        cfg = Mask2FormerConfig.from_pretrained(
            "facebook/mask2former-swin-large-coco-panoptic",
            num_labels=1,
            id2label={0: "filament"},
            label2id={"filament": 0},
        )
        # Pre-inizializza con pesi ufficiali per consistenza totale
        model = Mask2FormerForUniversalSegmentation.from_pretrained(
            "facebook/mask2former-swin-large-coco-panoptic",
            config=cfg,
            ignore_mismatched_sizes=True,
        )
        
        try:
            ckpt = torch.load(str(m2f_path), map_location="cpu", weights_only=True)
        except Exception:
            ckpt = torch.load(str(m2f_path), map_location="cpu", weights_only=False)
            
        state = ckpt
        for key in ("state_dict", "model_state_dict", "model"):
            if isinstance(ckpt, dict) and isinstance(ckpt.get(key), dict):
                state = ckpt[key]
                break
                
        ref_state = model.state_dict()
        clean_state = {}

        for k, v in state.items():
            # 1. Rimuove SOLO il prefisso iniziale DataParallel/DDP preservando i moduli interni
            k_clean = k[7:] if k.startswith("module.") else k

            # 2. Corrispondenza diretta o con model.
            if k_clean in ref_state and ref_state[k_clean].shape == v.shape:
                clean_state[k_clean] = v
                continue
            elif ("model." + k_clean) in ref_state and ref_state["model." + k_clean].shape == v.shape:
                clean_state["model." + k_clean] = v
                continue
            elif k_clean.startswith("model.") and k_clean[6:] in ref_state and ref_state[k_clean[6:]].shape == v.shape:
                clean_state[k_clean[6:]] = v
                continue

            # 3. Mappatura automatica Swin backbone legacy -> Transformers SwinBackbone moderno
            mapped_k = k_clean
            if "pixel_level_module.encoder.swin." in mapped_k:
                mapped_k = mapped_k.replace("pixel_level_module.encoder.swin.", "pixel_level_module.encoder.")
                mapped_k = mapped_k.replace(".attention.q_proj.", ".attention.self.query.")
                mapped_k = mapped_k.replace(".attention.k_proj.", ".attention.self.key.")
                mapped_k = mapped_k.replace(".attention.v_proj.", ".attention.self.value.")
                mapped_k = mapped_k.replace(".attention.o_proj.", ".attention.output.dense.")
                mapped_k = mapped_k.replace(
                    ".attention.relative_position_bias.relative_position_bias_table",
                    ".attention.self.relative_position_bias_table"
                )
                mapped_k = mapped_k.replace(".mlp.fc1.", ".intermediate.dense.")
                mapped_k = mapped_k.replace(".mlp.fc2.", ".output.dense.")

            if mapped_k in ref_state and ref_state[mapped_k].shape == v.shape:
                clean_state[mapped_k] = v
            elif ("model." + mapped_k) in ref_state and ref_state["model." + mapped_k].shape == v.shape:
                clean_state["model." + mapped_k] = v
            elif mapped_k.startswith("model.") and mapped_k[6:] in ref_state and ref_state[mapped_k[6:]].shape == v.shape:
                clean_state[mapped_k[6:]] = v

        model.load_state_dict(clean_state, strict=False)
        print(f"   ✅ Mask2Former caricato con successo: {len(clean_state)}/{len(state)} pesi validati!")
        return model.to(device).eval(), "MASK2FORMER"
    else:
        raise ValueError(f"Detector non valido: {detector_type}")


# Istanziazione effettiva dei modelli su singolo o multi-GPU (es. Kaggle 2x T4)
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_MULTI_GPU = (num_gpus >= 2)

workers = []
if USE_MULTI_GPU:
    print("=" * 75)
    print(f"⚡ INIZIALIZZAZIONE PIPELINE MULTI-GPU ({num_gpus} GPU PARALLELE)...")
    for g_idx in range(num_gpus):
        g_dev = torch.device(f"cuda:{g_idx}")
        print(f"\n--- [GPU {g_idx} - {g_dev}] Caricamento Modelli ---")
        s1, s1_t = load_stage1_model(STAGE1_DETECTOR, rfdetr_ckpt, m2f_ckpt, g_dev)
        ref, ref_ch = load_refiner_model(refiner_ckpt, g_dev)
        workers.append({
            "gpu_id": g_idx,
            "device": g_dev,
            "stage1_model": s1,
            "stage1_type": s1_t,
            "refiner_model": ref,
            "refiner_in_ch": ref_ch,
        })
    # Assegna worker 0 come riferimento globale per retrocompatibilità
    stage1_model = workers[0]["stage1_model"]
    stage1_type = workers[0]["stage1_type"]
    refiner_model = workers[0]["refiner_model"]
    refiner_in_ch = workers[0]["refiner_in_ch"]
    print("=" * 75)
    print(f"🚀 PIPELINE PRONTA SU TUTTE LE {num_gpus} GPU! L'inferenza sarà eseguita in parallelo.")
    print("=" * 75)
else:
    stage1_model, stage1_type = load_stage1_model(STAGE1_DETECTOR, rfdetr_ckpt, m2f_ckpt, device)
    refiner_model, refiner_in_ch = load_refiner_model(refiner_ckpt, device)
    workers.append({
        "gpu_id": 0,
        "device": device,
        "stage1_model": stage1_model,
        "stage1_type": stage1_type,
        "refiner_model": refiner_model,
        "refiner_in_ch": refiner_in_ch,
    })
    stage1_dev = next(stage1_model.parameters()).device if hasattr(stage1_model, "parameters") else device
    refiner_dev = next(refiner_model.parameters()).device if hasattr(refiner_model, "parameters") else device
    print("=" * 75)
    if str(stage1_dev).startswith("cuda") and str(refiner_dev).startswith("cuda"):
        print(f"🚀 ENTRAMBI I MODELLI SONO PRONTI IN MEMORIA GPU ({stage1_dev}) PER L'INFERENZA!")
    else:
        print(f"⚠️ ATTENZIONE: Modelli allocati su Stage1: {stage1_dev} | Refiner: {refiner_dev}.")
        print("   Consigliato attivare GPU T4 x 2 nelle impostazioni Kaggle!")
    print("=" * 75)

## 📐 Geometric Crop Pipeline: Two-Stage Decoupled Flow
### Integration of RF-DETR-Seg-Large (Stage 1) & Native-Resolution U-Net Refiner ResNet-18 (Stage 2)

```
[1] NATIVE FULL-DISK SOLAR IMAGE (2048 x 2048 px)
     │
     │ Stage 1 (RF-DETR @ 768px on 4 tiles): filament masks mapped back to global canvas.
     │ Masks with positive overlap (even across adjacent tiles) -> single group/task.
     │ Example: union bbox 81 x 125 px -> crop side = ceil(125 * 1.8) = 225 px (min 96)
     ▼
[2] NATIVE SQUARE CROP (225 x 225 px), shifted but never squished to stay within image bounds
     │
     │ cv2.resize(crop, (256, 256), cv2.INTER_LINEAR)
     ▼
[3] U-NET REFINER INPUT: batch (N, 3, 256, 256) in [0, 1], fp32 (ImageNet mean/std inside model)
     │
     │ CropUNet ResNet-18 with Flip-4 TTA (with topological guardrails per crop)
     ▼
[4] U-NET OUTPUT: probability map (256 x 256)
     │
     │ cv2.resize(prob_map, (225, 225), cv2.INTER_LINEAR)
     │ threshold >= 0.70; if no pixel exceeds threshold -> fallback to detector mask
     │ reinsertion into native coordinates [y0:y1, x0:x1]
     ▼
[5] MERGE OVERLAPPING TASKS -> disjoint filament instances in the solar disk (2048 x 2048 px)
```


## 🎯 5. Stage 1 Engine: Detector Masks to Refinement Tasks (Context 1.8x)
Process detector tile masks, map them onto the global $2048 \times 2048$ canvas, group positive-overlap masks across tiles into unified tasks, and compute dynamic native square crops with $1.8\times$ context factor.

In [ ]:
# ==============================================================================
# 5. STAGE 1 ENGINE: DETECTOR MASKS -> REFINEMENT TASKS (CONTEXT 1.8)
# ==============================================================================

def encode_mask(mask):
    """Maschera booleana/uint8 HxW -> RLE COCO non compresso-testo (dict con counts in bytes)."""
    return mask_utils.encode(np.asfortranarray(np.asarray(mask, dtype=np.uint8)))


def positive_overlap_groups(rles):
    """
    Raggruppa le maschere (RLE) che condividono almeno un pixel: componenti connesse del
    grafo di sovrapposizione. Un filamento rilevato da due tile adiacenti (overlap centrale
    di 152px) finisce nello stesso gruppo, e quindi in un solo task di raffinamento.
    """
    n = len(rles)
    if n < 2:
        return [[i] for i in range(n)]
    adjacency = np.asarray(mask_utils.iou(rles, rles, [0] * n), dtype=np.float64) > 0
    groups, seen = [], set()
    for start in range(n):
        if start in seen:
            continue
        stack, group = [start], []
        while stack:
            idx = stack.pop()
            if idx in seen:
                continue
            seen.add(idx)
            group.append(idx)
            stack.extend(np.flatnonzero(adjacency[idx]).tolist())
        groups.append(sorted(group))
    return groups


def merge_positive_overlap_rles(rles):
    """Unisce le maschere con sovrapposizione positiva: l'output e' disgiunto per costruzione."""
    if len(rles) < 2:
        return list(rles)
    return [mask_utils.merge([rles[i] for i in g]) for g in positive_overlap_groups(rles)]


def square_bounds_from_bbox(x, y, w, h, H=2048, W=2048, context=1.8, minimum=96):
    """
    Crop quadrato con fattore di contesto e lato minimo, centrato sulla bbox (x, y, w, h)
    e spostato (non ridimensionato) per restare dentro l'immagine: lato sempre = x1 - x0.
    """
    cx = x + w / 2.0
    cy = y + h / 2.0
    side = int(np.ceil(max(w, h) * context))
    side = min(max(side, minimum), min(H, W))

    x0 = int(round(cx - side / 2.0))
    y0 = int(round(cy - side / 2.0))
    x1 = x0 + side
    y1 = y0 + side

    if x0 < 0:
        x1 -= x0
        x0 = 0
    if y0 < 0:
        y1 -= y0
        y0 = 0
    if x1 > W:
        x0 -= (x1 - W)
        x1 = W
    if y1 > H:
        y0 -= (y1 - H)
        y1 = H
    return int(x0), int(y0), int(x1), int(y1)


def _to_numpy(x):
    if x is None:
        return None
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    return np.asarray(x)


def _place_tile_mask(tile_mask, x1, y1, x2, y2, H, W):
    """Riporta una maschera di tile sul canvas globale (resize nearest se la risoluzione differisce)."""
    tile_h, tile_w = y2 - y1, x2 - x1
    m = np.asarray(tile_mask).astype(np.uint8)
    if m.shape != (tile_h, tile_w):
        m = cv2.resize(m, (tile_w, tile_h), interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros((H, W), dtype=np.uint8)
    canvas[y1:y2, x1:x2] = m
    return canvas


def _rfdetr_predict(model, patch_rgb, conf_thresh):
    """predict() con fallback PIL -> ndarray e senza include_source_image per versioni vecchie di rfdetr."""
    errors = []
    for image in (Image.fromarray(patch_rgb), patch_rgb):
        for extra in ({"include_source_image": False}, {}):
            try:
                return model.predict(image, threshold=conf_thresh, **extra)
            except Exception as exc:
                errors.append(repr(exc))
    raise RuntimeError("RF-DETR predict() fallito: " + " | ".join(errors[:2]))


def _tile_detections_rfdetr(model, patch_rgb, conf_thresh):
    """Restituisce [(maschera_tile, score), ...] per un tile gia' ridimensionato a target_res."""
    res = _rfdetr_predict(model, patch_rgb, conf_thresh)
    if isinstance(res, (list, tuple)):
        res = res[0] if len(res) else None
    if res is None:
        return []
    xyxy = _to_numpy(getattr(res, "xyxy", None))
    if xyxy is None or len(xyxy) == 0:
        return []
    scores = _to_numpy(getattr(res, "confidence", None))
    if scores is None:
        scores = _to_numpy(getattr(res, "scores", None))
    masks = _to_numpy(getattr(res, "mask", None))
    if masks is None:
        raise RuntimeError("RF-DETR non ha restituito maschere (Detections.mask e' None): serve un checkpoint *Seg*.")
    if masks.ndim == 4:
        masks = masks[:, 0]

    out = []
    for i in range(len(xyxy)):
        score = float(scores[i]) if (scores is not None and len(scores) > i) else 0.80
        if score < conf_thresh:
            continue
        out.append((masks[i], score))
    return out


def _tile_detections_m2f(model, patch_rgb, target_res, conf_thresh, device):
    dev = torch.device(device)
    patch_norm = (patch_rgb.astype(np.float32) / 255.0 - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    tensor_in = torch.from_numpy(patch_norm.transpose(2, 0, 1)).float().unsqueeze(0).to(dev)

    with torch.inference_mode():
        with torch.amp.autocast(device_type="cuda" if dev.type == "cuda" else "cpu", dtype=torch.float16 if dev.type == "cuda" else torch.float32):
            out = model(pixel_values=tensor_in)

    probs = torch.softmax(out.class_queries_logits[0], dim=-1)[:, 0]
    valid_indices = torch.where(probs >= conf_thresh)[0]
    if len(valid_indices) == 0:
        return []

    # Interpola SOLO i query che superano la soglia per risparmiare memoria e tempo GPU/CPU
    sub_masks = out.masks_queries_logits[0, valid_indices].unsqueeze(0)
    mask_probs = torch.sigmoid(
        F.interpolate(sub_masks, size=(target_res, target_res), mode="bilinear", align_corners=False)
    ).squeeze(0)
    if mask_probs.ndim == 2:
        mask_probs = mask_probs.unsqueeze(0)

    result = []
    for i, idx in enumerate(valid_indices):
        bin_m = (mask_probs[i] > DETECTOR_MASK_THRESH).cpu().numpy().astype(np.uint8)
        if np.count_nonzero(bin_m) < 15:
            continue
        result.append((bin_m, float(probs[idx])))
    return result


def detect_filament_masks(model, model_type, img_gray, tile_windows, target_res=768, conf_thresh=0.41, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    """
    Stage 1: esegue il detector sui 4 tile e riporta ogni maschera sul canvas globale 2048x2048.
    Restituisce [{"rle": RLE, "score": float}, ...]; le maschere non vengono fuse qui.
    """
    H, W = img_gray.shape[:2]
    detections = []
    for (x1, y1, x2, y2) in tile_windows:
        patch = img_gray[y1:y2, x1:x2]
        patch_res = cv2.resize(patch, (target_res, target_res), interpolation=cv2.INTER_AREA)
        patch_rgb = cv2.cvtColor(patch_res, cv2.COLOR_GRAY2RGB)

        if model_type == "RF-DETR":
            tile_dets = _tile_detections_rfdetr(model, patch_rgb, conf_thresh)
        elif model_type == "MASK2FORMER":
            tile_dets = _tile_detections_m2f(model, patch_rgb, target_res, conf_thresh, device)
        else:
            raise ValueError(f"Detector non valido: {model_type}")

        for tile_mask, score in tile_dets:
            canvas = _place_tile_mask(tile_mask, x1, y1, x2, y2, H, W)
            if not canvas.any():
                continue
            detections.append({"rle": encode_mask(canvas), "score": float(score)})
    return detections


def build_refiner_tasks(detections, H=2048, W=2048, context=1.8, minimum=96):
    """
    Un task per ogni gruppo di maschere del detector che si sovrappongono (positivo overlap):
    la maschera del gruppo e' l'unione (usata anche come fallback se il refiner non produce nulla)
    e il crop quadrato e' calcolato sulla bbox esatta dell'unione.
    """
    rles = [d["rle"] for d in detections]
    tasks = []
    for group in positive_overlap_groups(rles):
        union_rle = mask_utils.merge([rles[i] for i in group])
        x, y, w, h = [float(v) for v in mask_utils.toBbox(union_rle)]
        x0, y0, x1, y1 = square_bounds_from_bbox(x, y, w, h, H=H, W=W, context=context, minimum=minimum)
        tasks.append({
            "bounds": (x0, y0, x1, y1),
            "side": x1 - x0,
            "fallback_rle": union_rle,
            "orig_box": (int(x), int(y), int(np.ceil(x + w)), int(np.ceil(y + h))),
            "score": max(detections[i]["score"] for i in group),
            "n_members": len(group),
        })
    return tasks


## 🔬 6. Stage 2 Engine: Native-Resolution Refiner & Topology-Safe Flip-4 TTA
Perform batched inference on $256 \times 256$ crops using `CropUNet` with 4-view test-time augmentation (identity, horizontal, vertical, and combined flips) and topological guardrails. Rescale continuous probability maps back to native crop dimensions and apply threshold $\ge 0.70$.

In [ ]:
# ==============================================================================
# 6. STAGE 2 ENGINE: NATIVE-RESOLUTION REFINER & TOPOLOGY-SAFE FLIP-4 TTA
# ==============================================================================

REFINER_TTA_TRANSFORMS = ("identity", "hflip", "vflip", "hvflip")


def _flip_batch(arr, transform):
    """Flip di un batch [N, H, W]. Ogni flip e' un'involuzione: lo stesso codice applica e inverte."""
    if transform == "identity":
        return np.ascontiguousarray(arr)
    if transform == "hflip":
        return np.ascontiguousarray(arr[:, :, ::-1])
    if transform == "vflip":
        return np.ascontiguousarray(arr[:, ::-1, :])
    if transform == "hvflip":
        return np.ascontiguousarray(arr[:, ::-1, ::-1])
    raise ValueError(f"Trasformazione TTA sconosciuta: {transform}")


def _run_refiner_batches(refiner, crops, device, batch_size=32):
    """
    Forward del refiner in fp32 su crop uint8 [N, S, S] in scala 0-255 (il modello normalizza
    internamente con mean/std ImageNet). Restituisce le probabilita' [N, S, S] float32.
    """
    size = REFINER_CROP_SIZE
    if crops.ndim != 3 or crops.shape[1:] != (size, size):
        raise ValueError(f"I crop del refiner devono avere shape [N, {size}, {size}], trovato {crops.shape}")
    outputs = []
    with torch.inference_mode():
        for start in range(0, len(crops), batch_size):
            chunk = np.ascontiguousarray(crops[start:start + batch_size])
            batch = (
                torch.from_numpy(chunk)
                .float()
                .unsqueeze(1)
                .repeat(1, 3, 1, 1)
                .to(device, non_blocking=True)
                / 255
            )
            outputs.append(refiner(batch).sigmoid().cpu().numpy()[:, 0].astype(np.float32))
    return np.concatenate(outputs, axis=0)


def _component_count_8(mask):
    return int(cv2.connectedComponents(np.asarray(mask, dtype=np.uint8), connectivity=8)[0] - 1)


def _mask_dice(first, second):
    first = np.asarray(first, dtype=bool)
    second = np.asarray(second, dtype=bool)
    denominator = int(first.sum() + second.sum())
    if denominator == 0:
        return 1.0
    return 2.0 * int(np.logical_and(first, second).sum()) / denominator


def select_topology_safe_tta_probability(single_probability, tta_probability, threshold):
    """
    Usa la mappa TTA solo se la topologia della maschera binarizzata resta vicina a quella
    della vista singola. Restituisce (mappa scelta, accettata?, motivo).
    """
    single_mask = np.asarray(single_probability, dtype=np.float32) >= np.float32(threshold)
    tta_mask = np.asarray(tta_probability, dtype=np.float32) >= np.float32(threshold)
    single_area = int(single_mask.sum())
    tta_area = int(tta_mask.sum())
    if tta_area == 0:
        return single_probability, False, "empty_tta_mask"
    if single_area == 0:
        return single_probability, False, "empty_single_view_mask"
    area_ratio = tta_area / max(single_area, 1)
    if area_ratio < TTA_AREA_RATIO_MIN:
        return single_probability, False, "area_shrink_guard"
    if area_ratio > TTA_AREA_RATIO_MAX:
        return single_probability, False, "area_growth_guard"
    if _component_count_8(tta_mask) > max(_component_count_8(single_mask) + 1, 2):
        return single_probability, False, "fragmentation_guard"
    if single_area >= TTA_MIN_CONTROL_AREA_FOR_DICE:
        if _mask_dice(single_mask, tta_mask) < TTA_DICE_MIN:
            return single_probability, False, "overlap_guard"
    return tta_probability, True, "accepted"


def infer_refiner_probabilities(refiner, crops, device, use_tta=True, threshold=0.70, batch_size=32):
    """
    Probabilita' del refiner per tutti i crop dell'immagine (batch unico per vista).
    Con use_tta=True media le 4 viste (identity/h/v/hv) ma, crop per crop, ricade sulla
    vista singola se un guardrail topologico scatta. Restituisce (probabilita' [N,S,S], decisioni).
    """
    if not use_tta:
        single = _run_refiner_batches(refiner, crops, device, batch_size)
        return single, [{"accepted": False, "reason": "tta_disabled"} for _ in range(len(single))]

    single, accumulator = None, None
    for transform in REFINER_TTA_TRANSFORMS:
        probs = _run_refiner_batches(refiner, _flip_batch(crops, transform), device, batch_size)
        probs = _flip_batch(probs, transform)
        if transform == "identity":
            single = probs.astype(np.float32, copy=True)
        accumulator = probs.astype(np.float32, copy=True) if accumulator is None else accumulator + probs
    mean_probs = (accumulator / np.float32(len(REFINER_TTA_TRANSFORMS))).astype(np.float32)

    selected, decisions = [], []
    for single_p, tta_p in zip(single, mean_probs):
        chosen, accepted, reason = select_topology_safe_tta_probability(single_p, tta_p, threshold)
        selected.append(np.asarray(chosen, dtype=np.float32))
        decisions.append({"accepted": bool(accepted), "reason": str(reason)})
    return np.stack(selected).astype(np.float32), decisions


def decode_task_mask(probability, bounds, fallback_rle, threshold, H=2048, W=2048):
    """
    Probabilita' 256x256 -> maschera nativa 2048x2048 (resize bilineare della probabilita' continua,
    poi soglia). Se il refiner non produce pixel, ripiega sulla maschera del detector.
    Restituisce (maschera bool HxW, usato_fallback).
    """
    x0, y0, x1, y1 = bounds
    crop_mask = cv2.resize(
        np.asarray(probability, dtype=np.float32), (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR
    ) >= np.float32(threshold)
    if not crop_mask.any():
        return mask_utils.decode(fallback_rle).astype(bool), True
    full = np.zeros((H, W), dtype=bool)
    full[y0:y1, x0:x1] = crop_mask
    return full, False


## 🧹 7. Post-Processing: Morphological Anti-Dot Filter (Optional)
Optional post-processing utility to filter out small, circular spurious artifacts (dots) while strictly preserving thin, elongated solar filament structures.

In [ ]:
# ==============================================================================
# 7. POST-PROCESSING: MORPHOLOGICAL ANTI-DOT FILTER (OPTIONAL)
# ==============================================================================

def filter_anti_dots(mask_bool, max_dot_area=120, min_aspect_ratio=1.60, max_circ=0.70):
    """
    Rimuove dall'istanza le componenti connesse che sono "pallini": area < max_dot_area E forma
    compatta (aspect ratio < min_aspect_ratio e circolarita' > max_circ). Le componenti piccole
    ma allungate (punte, frammenti sottili di filamento) vengono mantenute.
    """
    if not np.any(mask_bool):
        return mask_bool

    mask_u8 = np.asarray(mask_bool, dtype=np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    clean_mask = np.zeros(mask_u8.shape, dtype=bool)

    for lbl in range(1, num_labels):
        x, y, w, h, area = (int(stats[lbl, c]) for c in (
            cv2.CC_STAT_LEFT, cv2.CC_STAT_TOP, cv2.CC_STAT_WIDTH, cv2.CC_STAT_HEIGHT, cv2.CC_STAT_AREA))
        roi = labels[y:y + h, x:x + w] == lbl

        if area < max_dot_area:
            cnts, _ = cv2.findContours(roi.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if cnts:
                cnt = max(cnts, key=cv2.contourArea) if len(cnts) > 1 else cnts[0]
                if len(cnt) >= 5:
                    (w_rect, h_rect) = cv2.minAreaRect(cnt)[1]
                    aspect_ratio = max(w_rect, h_rect) / (min(w_rect, h_rect) + 1e-5)
                else:
                    aspect_ratio = max(w, h) / (min(w, h) + 1e-5)
                perimeter = cv2.arcLength(cnt, True)
                circularity = 4 * np.pi * area / (perimeter ** 2 + 1e-5)
                if aspect_ratio < min_aspect_ratio and circularity > max_circ:
                    continue

        clean_mask[y:y + h, x:x + w] |= roi

    return clean_mask


## 🚀 8. End-to-End Single-Image Processing Pipeline & Multi-GPU Dispatcher
Orchestrate the full inference pipeline for an individual $2048 \times 2048$ solar image: Stage 1 detection, task clustering, Stage 2 refinement, fallback mechanisms, non-overlapping instance merging, and spatial sorting. Includes an optimized multi-GPU dispatcher for dual T4 setups.

In [ ]:
# ==============================================================================
# 8. FULL SINGLE-IMAGE PIPELINE: DETECTOR -> TASK -> REFINER -> MERGE
# ==============================================================================

def process_single_image(img_path, stage1_model, stage1_type, refiner_model, refiner_in_ch, device, debug=False):
    """
    Elabora un'immagine solare 2048x2048 attraverso i 2 stadi disaccoppiati.
    Restituisce una lista di maschere booleane 2048x2048 a sovrapposizione ZERO, ordinate per (y, x).
    """
    img_gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        raise RuntimeError(f"Immagine non leggibile: {img_path}")
    H, W = img_gray.shape[:2]
    if (H, W) != (2048, 2048):
        raise ValueError(f"Attesa immagine 2048x2048, trovata {W}x{H}: {img_path}")
    stem = Path(img_path).stem

    # STAGE 1: maschere del detector sui 4 tile, riportate su canvas globale
    detections = detect_filament_masks(
        stage1_model, stage1_type, img_gray, TILE_WINDOWS,
        target_res=DETECTOR_TILE_RES, conf_thresh=DETECTOR_CONF_THRESH, device=device
    )
    if not detections:
        if debug:
            print(f"  🔍 [DEBUG {stem}] Stage 1 ({stage1_type}): nessun filamento rilevato.")
        return []

    # Un task per gruppo di maschere sovrapposte (duplicati sulle giunzioni dei tile inclusi)
    tasks = build_refiner_tasks(detections, H=H, W=W, context=CROP_CONTEXT, minimum=CROP_MIN_SIDE)

    # STAGE 2: crop nativi quadrati -> 256x256 -> refiner (fp32) con Flip-4 TTA topology-safe
    crops = np.stack([
        cv2.resize(img_gray[y0:y1, x0:x1], (REFINER_CROP_SIZE, REFINER_CROP_SIZE), interpolation=cv2.INTER_LINEAR)
        for (x0, y0, x1, y1) in (t["bounds"] for t in tasks)
    ])
    probs, decisions = infer_refiner_probabilities(
        refiner_model, crops, device, use_tta=ENABLE_TOPOLOGY_SAFE_TTA,
        threshold=REFINER_PROB_THRESH, batch_size=REFINER_BATCH_SIZE
    )

    task_rles, n_fallback = [], 0
    for task, prob in zip(tasks, probs):
        mask, used_fallback = decode_task_mask(prob, task["bounds"], task["fallback_rle"], REFINER_PROB_THRESH, H, W)
        n_fallback += int(used_fallback)
        task_rles.append(encode_mask(mask))

    # Merge dei task che si sovrappongono: l'output e' disgiunto per costruzione (nessun pixel in due istanze)
    merged = merge_positive_overlap_rles(task_rles)
    merged.sort(key=lambda r: (float(mask_utils.toBbox(r)[1]), float(mask_utils.toBbox(r)[0])))

    instances = []
    for rle in merged:
        inst = mask_utils.decode(rle).astype(bool)
        if ENABLE_ANTI_PALLINI:
            inst = filter_anti_dots(inst, max_dot_area=DOT_MAX_AREA_PX,
                                    min_aspect_ratio=MIN_ASPECT_RATIO, max_circ=MAX_CIRCULARITY)
        if np.count_nonzero(inst) >= MIN_INSTANCE_AREA_PX:
            instances.append(inst)

    if debug:
        reasons = defaultdict(int)
        for d in decisions:
            reasons[d["reason"]] += 1
        print(f"  🔍 [DEBUG {stem}] {len(detections)} maschere detector -> {len(tasks)} task | "
              f"TTA: {dict(reasons)} | fallback detector: {n_fallback} | istanze finali: {len(instances)}")

    return instances


# ==============================================================================
# MULTI-GPU DISPATCHER OPTIMIZED FOR 2x T4 (DUAL-CHANNEL PARALLEL INFERENCE)
# ==============================================================================
import queue
from concurrent.futures import ThreadPoolExecutor

def process_images_parallel(img_paths, workers, desc="Elaborazione", debug_first_n=2):
    """
    Esegue l'inferenza della decoupled pipeline in parallelo sfruttando tutte le GPU disponibili.
    Con 2x T4 (Kaggle), processa 2 immagini simultaneamente raddoppiando il throughput.
    L'ordine delle immagini restituite è garantito identico al 100% rispetto alla lista di input.
    """
    n_workers = len(workers)
    n_images = len(img_paths)
    if n_images == 0:
        return []

    # Se c'è solo 1 GPU o CPU, esegue il percorso standard sequenziale
    if n_workers <= 1:
        w = workers[0]
        results = []
        for idx, img_p in enumerate(tqdm(img_paths, desc=desc)):
            debug_flag = (idx < debug_first_n)
            masks = process_single_image(
                img_p,
                stage1_model=w["stage1_model"],
                stage1_type=w["stage1_type"],
                refiner_model=w["refiner_model"],
                refiner_in_ch=w["refiner_in_ch"],
                device=w["device"],
                debug=debug_flag
            )
            results.append((img_p, masks))
        return results

    # Coda thread-safe per i worker GPU
    worker_q = queue.Queue()
    for w in workers:
        worker_q.put(w)

    def _worker_step(item):
        idx, img_p = item
        w = worker_q.get()
        debug_flag = (idx < debug_first_n)
        try:
            masks = process_single_image(
                img_p,
                stage1_model=w["stage1_model"],
                stage1_type=w["stage1_type"],
                refiner_model=w["refiner_model"],
                refiner_in_ch=w["refiner_in_ch"],
                device=w["device"],
                debug=debug_flag
            )
            return idx, img_p, masks
        finally:
            worker_q.put(w)

    results = [None] * n_images
    desc_str = f"{desc} (x{n_workers} GPU)"
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        for idx, img_p, masks in tqdm(executor.map(_worker_step, enumerate(img_paths)), total=n_images, desc=desc_str):
            results[idx] = (img_p, masks)

    return results


## 📊 8.bis Self-Evaluation: Panoptic Quality (PQ) Validation & Comparative Gallery
Execute comprehensive self-evaluation across the validation dataset, calculating Panoptic Quality (PQ), Segmentation Quality (SQ), Recognition Quality (RQ), and mean Dice score (0.741 on validation check), with multi-panel visual comparison galleries.

In [ ]:
# ==============================================================================
# 8.bis (SELF-EVALUATION) PANOPTIC QUALITY (PQ) VALIDATION ON FULL VAL SET
#       WITH COMPARATIVE VISUAL INSPECTION GALLERY ON FIRST 4 SAMPLE IMAGES
# ==============================================================================
import os
import glob
import time
import json
import random
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pycocotools import mask as mask_utils
from IPython.display import FileLink, display

print("=" * 85)
print("🚀 AVVIO SELF-EVALUATION SU TUTTO IL SET DI VALIDAZIONE (AZIM AHMADZADEH FORMAT)")
print("=" * 85)

# --- Validation Configuration ---
MAX_VAL_IMAGES = None  # None valuta tutte le 106 immagini dello split di validazione
IOU_THRESHOLD  = 0.50

# ------------------------------------------------------------------------------
# 1. RICERCA AUTOMATICA ANNOTAZIONI COCO TRAIN & CARTELLA IMMAGINI
# ------------------------------------------------------------------------------
search_roots = [
    Path("/kaggle/input/competitions/filament-segmentation-2026"),
    Path("/kaggle/input/filament-segmentation-2026"),
    Path("/kaggle/input/datasets/esairlab/magfilo-v1-0-segmentation-of-solar-filaments"),
    Path("/kaggle/input/magfilo-v1-0-segmentation-of-solar-filaments"),
    Path("/kaggle/input"),
    Path(".")
]

train_json_path = None
train_images_dir = None

for root in search_roots:
    if root.exists():
        for cand_json in root.rglob("*.json"):
            if "train" in cand_json.name.lower():
                parent = cand_json.parent
                for cand_img in [
                    parent / "train_images",
                    parent / "train" / "train_images",
                    parent / "train",
                    parent / "images"
                ]:
                    if cand_img.exists() and any(cand_img.glob("*.*")):
                        train_json_path = cand_json
                        train_images_dir = cand_img
                        break
            if train_json_path and train_images_dir:
                break
    if train_json_path and train_images_dir:
        break

if not train_json_path or not train_images_dir:
    print("❌ Dataset train non trovato. Verifica i percorsi input.")
else:
    print(f"✅ Annotazioni COCO trovate : {train_json_path}")
    print(f"✅ Cartella immagini train : {train_images_dir}")

    # --------------------------------------------------------------------------
    # 2. SPLIT DETERMINISTICO VALIDAZIONE 15% (106 IMMAGINI)
    # --------------------------------------------------------------------------
    with open(train_json_path, "r", encoding="utf-8") as f:
        coco_orig = json.load(f)
        
    file_to_records = defaultdict(list)
    for img in coco_orig["images"]:
        file_to_records[img["file_name"]].append(img)
        
    unique_files = sorted(list(file_to_records.keys()))
    random.seed(42)
    random.shuffle(unique_files)
    
    n_val = max(1, int(len(unique_files) * 0.15))
    val_files = sorted(unique_files[:n_val])
    if MAX_VAL_IMAGES is not None and MAX_VAL_IMAGES > 0:
        val_files = val_files[:MAX_VAL_IMAGES]
        
    image_id_to_anns = defaultdict(list)
    for ann in coco_orig["annotations"]:
        image_id_to_anns[ann["image_id"]].append(ann)
        
    print(f"📊 Immagini selezionate per la validazione: {len(val_files)}")

    # --------------------------------------------------------------------------
    # 3. ESTRAZIONE GROUND TRUTH MULTI-ANNOTATORE (FORMATO AZIM AHMADZADEH)
    # --------------------------------------------------------------------------
    print("\n⏳ [1/3] Estrazione maschere Ground-Truth...")
    gt_rows = []
    gt_by_img_annotator = defaultdict(lambda: defaultdict(list))

    for fn in val_files:
        records = file_to_records[fn]
        img_stem = Path(fn).stem
        for annotator_idx, img_rec in enumerate(records, start=1):
            img_id = img_rec["id"]
            # Separazione corretta di ciascun annotatore indipendente
            if "-" in str(img_id):
                annotator_name = str(img_id).split("-", 1)[0]
            else:
                annotator_name = str(img_rec.get("annotator_id", annotator_idx))

            anns = image_id_to_anns[img_id]
            mask_idx = 0
            for ann in anns:
                seg = ann.get("segmentation", None)
                if not seg:
                    continue
                rle_dict = mask_utils.frPyObjects(seg, 2048, 2048)
                if isinstance(rle_dict, list):
                    rle_dict = mask_utils.merge(rle_dict)
                counts_str = rle_dict["counts"].decode("utf-8") if isinstance(rle_dict["counts"], bytes) else str(rle_dict["counts"])
                counts_str = counts_str.strip().replace("\n", "").replace("\r", "")
                
                gt_rows.append({
                    "filament_id": f"{annotator_name}-{img_stem}_{mask_idx}",
                    "segmentation_rle": counts_str
                })
                gt_by_img_annotator[img_stem][annotator_name].append(counts_str)
                mask_idx += 1

    val_gt_df = pd.DataFrame(gt_rows)
    val_gt_csv = Path("/kaggle/working/val_ground_truth.csv")
    val_gt_df.to_csv(val_gt_csv, index=False)
    print(f"✅ Salvato: {val_gt_csv} ({len(val_gt_df)} annotazioni ground-truth)")

    # --------------------------------------------------------------------------
    # 4. INFERENZA PIPELINE DISACCOPPIATA SU TUTTE LE IMMAGINI DI VALIDAZIONE
    # --------------------------------------------------------------------------
    print(f"\n⏳ [2/3] Esecuzione Pipeline Disaccoppiata su {len(val_files)} immagini...")
    val_img_paths = [train_images_dir / f for f in val_files if (train_images_dir / f).exists()]
    pred_rows = []
    preds_by_img = defaultdict(list)
    raw_images_rgb = {}
    t_val_start = time.time()

    # Inferenza parallela ad alto throughput multi-GPU (supporta 1x GPU, 2x T4 o CPU)
    parallel_val_results = process_images_parallel(
        val_img_paths,
        workers=workers,
        desc="Validazione Decoupled Pipeline",
        debug_first_n=2
    )

    for idx, (img_p, masks) in enumerate(parallel_val_results):
        stem = img_p.stem
        
        # Salva in RAM solo le prime 4 immagini per la galleria grafica
        if idx < 4:
            img_bgr = cv2.imread(str(img_p))
            if img_bgr is not None:
                raw_images_rgb[stem] = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            if idx >= 2:
                print(f"  ↳ Campione {idx+1} [{stem}]: {len(masks)} filamenti estratti")

        count_m = 0
        for inst_mask in masks:
            fortran_m = np.asfortranarray(inst_mask.astype(np.uint8))
            rle_dict = mask_utils.encode(fortran_m)
            counts_str = rle_dict["counts"].decode("utf-8") if isinstance(rle_dict["counts"], bytes) else str(rle_dict["counts"])
            counts_str = counts_str.strip().replace("\n", "").replace("\r", "")
            
            pred_rows.append({
                "filament_id": f"{stem}_{count_m + 1}",
                "segmentation_rle": counts_str
            })
            preds_by_img[stem].append(counts_str)
            count_m += 1

    t_val_total = time.time() - t_val_start
    val_pred_df = pd.DataFrame(pred_rows)
    val_pred_csv = Path("/kaggle/working/val_predictions.csv")
    val_pred_df.to_csv(val_pred_csv, index=False)
    print(f"✅ Salvato: {val_pred_csv} ({len(val_pred_df)} filamenti predetti in {t_val_total:.2f}s, ~{t_val_total/len(val_img_paths):.2f}s/img)")

    # --------------------------------------------------------------------------
    # 5. CALCOLO METRICHE UFFICIALI PANOPTIC QUALITY (PQ, SQ, RQ)
    # --------------------------------------------------------------------------
    print("\n⏳ [3/3] Calcolo Panoptic Quality ufficiale su tutte le immagini (IoU >= 0.50)...")
    
    total_tp, total_fp, total_fn = 0, 0, 0
    total_iou = 0.0
    image_stats = {}
    matches_by_stem = {}

    for stem in [p.stem for p in val_img_paths]:
        pred_rles = preds_by_img.get(stem, [])
        pred_coco = [{"counts": r.encode("ascii") if isinstance(r, str) else r, "size": [2048, 2048]} for r in pred_rles]

        img_tp, img_fp, img_fn = 0, 0, 0
        img_iou = 0.0
        annotators = gt_by_img_annotator.get(stem, {})

        matched_preds_set = set()
        matched_gts_set = set()

        if not annotators:
            img_fp = len(pred_coco)
        else:
            for annotator, gt_rles in annotators.items():
                gt_coco = [{"counts": r.encode("ascii") if isinstance(r, str) else r, "size": [2048, 2048]} for r in gt_rles]

                if not pred_coco and not gt_coco:
                    continue
                if not pred_coco:
                    img_fn += len(gt_coco)
                    continue
                if not gt_coco:
                    img_fp += len(pred_coco)
                    continue

                ious = mask_utils.iou(pred_coco, gt_coco, [0] * len(gt_coco))

                pairs = []
                for p_i in range(len(pred_coco)):
                    for g_i in range(len(gt_coco)):
                        val = float(ious[p_i, g_i])
                        if val > IOU_THRESHOLD:
                            pairs.append((val, p_i, g_i))
                pairs.sort(key=lambda x: x[0], reverse=True)

                cur_matched_gt = set()
                cur_matched_pr = set()
                for val, p_i, g_i in pairs:
                    if p_i not in cur_matched_pr and g_i not in cur_matched_gt:
                        cur_matched_pr.add(p_i)
                        cur_matched_gt.add(g_i)
                        img_tp += 1
                        img_iou += val
                        matched_preds_set.add(p_i)
                        matched_gts_set.add(g_i)

                img_fp += len(pred_coco) - len(cur_matched_pr)
                img_fn += len(gt_coco) - len(cur_matched_gt)

        sq = img_iou / img_tp if img_tp > 0 else 0.0
        denom = img_tp + 0.5 * img_fp + 0.5 * img_fn
        rq = img_tp / denom if denom > 0 else 0.0
        pq = sq * rq

        gt_count = sum(len(rles) for rles in annotators.values()) if annotators else 0
        image_stats[stem] = {
            "gt_count": gt_count,
            "pred_count": len(pred_coco),
            "TP": img_tp,
            "FP": img_fp,
            "FN": img_fn,
            "SQ": sq,
            "RQ": rq,
            "PQ": pq
        }

        matches_by_stem[stem] = {
            "matched_preds": matched_preds_set,
            "matched_gts": matched_gts_set,
            "pred_coco": pred_coco,
            "annotators": annotators
        }

        total_tp += img_tp
        total_fp += img_fp
        total_fn += img_fn
        total_iou += img_iou

    glob_sq = total_iou / total_tp if total_tp > 0 else 0.0
    glob_denom = total_tp + 0.5 * total_fp + 0.5 * total_fn
    glob_rq = total_tp / glob_denom if glob_denom > 0 else 0.0
    glob_pq = glob_sq * glob_rq

    # Stampa Risultati Globali su tutto il validation set
    print("\n" + "=" * 85)
    print(f"🏆 RISULTATI DELLA VALIDAZIONE UFFICIALE ({len(val_img_paths)} IMMAGINI - PANOPTIC QUALITY):")
    print("=" * 85)
    print(f"🎯 PANOPTIC QUALITY  (PQ): {glob_pq:.4f}")
    print(f"📈 SEGMENTATION QUAL (SQ): {glob_sq:.4f}  (Accuratezza dei bordi IoU)")
    print(f"🎯 RECOGNITION QUAL  (RQ): {glob_rq:.4f}  (F1-score globale di rilevamento)")
    print(f"📋 METRICHE COUNTING     : TP = {total_tp}  |  FP = {total_fp}  |  FN = {total_fn}")
    print(f"⏱️ Tempo Validazione     : {t_val_total:.2f}s (~{t_val_total / len(val_img_paths):.2f}s per immagine)")
    print("=" * 85)

    # --------------------------------------------------------------------------
    # 6. TABELLA DETTAGLIATA SULLE PRIME 4 IMMAGINI CAMPIONE
    # --------------------------------------------------------------------------
    sample_stems = [p.stem for p in val_img_paths[:4]]
    print("\n" + "-" * 85)
    print("📊 DETTAGLIO METRICHE SULLE 4 IMMAGINI CAMPIONE MOSTRATE NEI GRAFICI:")
    print("-" * 85)
    header_str = f"{'ID Immagine':<20} | {'GT':<4} | {'Pred':<5} | {'TP':<4} | {'FP':<4} | {'FN':<4} | {'SQ (IoU)':<9} | {'RQ (F1)':<8} | {'PQ':<8}"
    print(header_str)
    print("-" * 85)
    for s in sample_stems:
        st = image_stats[s]
        row_str = f"{s:<20} | {st['gt_count']:<4} | {st['pred_count']:<5} | {st['TP']:<4} | {st['FP']:<4} | {st['FN']:<4} | {st['SQ']:<9.4f} | {st['RQ']:<8.4f} | {st['PQ']:<8.4f}"
        print(row_str)
    print("-" * 85)

    # --------------------------------------------------------------------------
    # 7. GALLERIA GRAFICA COMPARATIVA AD ALTA DEFINIZIONE (PRIME 4 IMMAGINI)
    # --------------------------------------------------------------------------
    print("\n🎨 Generazione galleria visiva a 4 pannelli per le 4 immagini campione...")
    fig, axes = plt.subplots(len(sample_stems), 4, figsize=(22, 5.5 * len(sample_stems)))
    if len(sample_stems) == 1:
        axes = np.expand_dims(axes, 0)

    palette = [
        (0, 255, 255),    # Ciano
        (255, 105, 180),  # Rosa
        (255, 215, 0),    # Oro
        (0, 191, 255),    # Celeste
        (186, 85, 211),   # Viola
        (255, 140, 0),    # Arancione
        (50, 205, 50),    # Verde lime
    ]

    for row_idx, stem in enumerate(sample_stems):
        st = image_stats[stem]
        img_rgb = raw_images_rgb.get(stem, np.zeros((2048, 2048, 3), dtype=np.uint8))

        # [1] Ground Truth Multi-Annotatore
        gt_panel = img_rgb.copy()
        gt_annotators = matches_by_stem[stem]["annotators"]
        inst_c = 0
        for ann_id, rles in gt_annotators.items():
            for r in rles:
                m = mask_utils.decode({"size": [2048, 2048], "counts": r.encode("ascii") if isinstance(r, str) else r})
                if np.any(m):
                    col = palette[inst_c % len(palette)]
                    gt_panel[m > 0] = (gt_panel[m > 0] * 0.45 + np.array(col) * 0.55).astype(np.uint8)
                    contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(gt_panel, contours, -1, col, 2)
                    inst_c += 1

        # [2] Predizione Decoupled
        pr_panel = img_rgb.copy()
        pred_coco = matches_by_stem[stem]["pred_coco"]
        for p_i, p_item in enumerate(pred_coco):
            m = mask_utils.decode(p_item)
            if np.any(m):
                col = palette[p_i % len(palette)]
                pr_panel[m > 0] = (pr_panel[m > 0] * 0.45 + np.array(col) * 0.55).astype(np.uint8)
                contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(pr_panel, contours, -1, col, 2)

        # [3] Diagnostica Metriche (Verde = TP, Rosso = FP, Arancione = FN)
        diag_panel = img_rgb.copy()
        matched_preds = matches_by_stem[stem]["matched_preds"]
        matched_gts = matches_by_stem[stem]["matched_gts"]

        # FN: Ground Truth non catturati (Arancione)
        for ann_id, rles in gt_annotators.items():
            for g_i, r in enumerate(rles):
                if g_i not in matched_gts:
                    m = mask_utils.decode({"size": [2048, 2048], "counts": r.encode("ascii") if isinstance(r, str) else r})
                    if np.any(m):
                        diag_panel[m > 0] = (diag_panel[m > 0] * 0.4 + np.array([255, 165, 0]) * 0.6).astype(np.uint8)
                        contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                        cv2.drawContours(diag_panel, contours, -1, (255, 165, 0), 2)

        # FP: Predizioni errate (Rosso)
        for p_i, p_item in enumerate(pred_coco):
            if p_i not in matched_preds:
                m = mask_utils.decode(p_item)
                if np.any(m):
                    diag_panel[m > 0] = (diag_panel[m > 0] * 0.35 + np.array([255, 50, 50]) * 0.65).astype(np.uint8)
                    contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(diag_panel, contours, -1, (255, 50, 50), 2)

        # TP: True Positives con IoU >= 0.50 (Verde brillante)
        for p_i in matched_preds:
            m = mask_utils.decode(pred_coco[p_i])
            if np.any(m):
                diag_panel[m > 0] = (diag_panel[m > 0] * 0.35 + np.array([50, 255, 50]) * 0.65).astype(np.uint8)
                contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(diag_panel, contours, -1, (50, 255, 50), 2)

        ax0, ax1, ax2, ax3 = axes[row_idx]

        ax0.imshow(img_rgb, cmap="gray")
        ax0.set_title(f"[1/4] Sole 2048x2048\nID: {stem}", fontsize=11, fontweight="bold")
        ax0.axis("off")

        ax1.imshow(gt_panel)
        ax1.set_title(f"[2/4] Ground Truth Multi-Annotatore\n({st['gt_count']} annotazioni)", fontsize=11, fontweight="bold", color="teal")
        ax1.axis("off")

        ax2.imshow(pr_panel)
        ax2.set_title(f"[3/4] Predizione U-Net\n({st['pred_count']} filamenti estratti)", fontsize=11, fontweight="bold", color="navy")
        ax2.axis("off")

        ax3.imshow(diag_panel)
        diag_color = "darkgreen" if st["PQ"] > 0.5 else "maroon"
        ax3.set_title(f"[4/4] PQ: {st['PQ']:.3f} (SQ: {st['SQ']:.2f}, RQ: {st['RQ']:.2f})\nTP={st['TP']} | FP={st['FP']} | FN={st['FN']}", fontsize=11, fontweight="bold", color=diag_color)
        ax3.axis("off")

    plt.tight_layout()
    plt.show()

    print("\n📥 DOWNLOAD FILE DI VALIDAZIONE PER CONFRONTO ESTERNO:")
    display(FileLink(str(val_gt_csv)))
    display(FileLink(str(val_pred_csv)))

    fig.savefig("/kaggle/working/validation_gallery.png", bbox_inches="tight", dpi=150)
    display(FileLink("/kaggle/working/validation_gallery.png"))

## ⚡ 9. Full Test Set Batch Inference & Compressed COCO RLE Encoding
Run concurrent multi-worker inference on all competition test images with progress tracking (`tqdm`), extract refined binary masks, and encode them into compressed COCO RLE format.

In [ ]:
# ==============================================================================
# 9. FULL TEST SET INFERENCE & COMPRESSED COCO RLE ENCODING
# ==============================================================================
from tqdm.auto import tqdm
from IPython.display import FileLink, display

print("=" * 75)
print(f"🚀 AVVIO INFERENZA UFFICIALE SU {len(test_files)} IMMAGINI DI TEST")
print(f"   Worker attivi: {len(workers)} GPU per elaborazione parallela concorrente")
print("=" * 75)

submission_rows = []
t_start_total = time.time()
total_filaments_found = 0

parallel_test_results = process_images_parallel(
    test_files,
    workers=workers,
    desc="Inferenza Test Set",
    debug_first_n=0
)

for img_p, masks in parallel_test_results:
    stem = img_p.stem
    count_img = 0
    for inst_mask in masks:
        if not inst_mask.any():
            continue
        # Garanzia Zero-Overlap: istanze già disgiunte per costruzione (validate_submission_df lo ricontrolla)
        fortran_m = np.asfortranarray(inst_mask.astype(np.uint8))
        rle_dict = mask_utils.encode(fortran_m)
        rle_str = rle_dict["counts"].decode("utf-8") if isinstance(rle_dict["counts"], bytes) else str(rle_dict["counts"])
        
        # Pulizia stringa da eventuali ritorni a capo o spazi
        rle_str = rle_str.strip().replace("\n", "").replace("\r", "")
        
        filament_id = f"{stem}_{count_img + 1}"
        submission_rows.append({
            "filament_id": filament_id,
            "segmentation_rle": rle_str
        })
        count_img += 1
        
    total_filaments_found += count_img

t_total_sec = time.time() - t_start_total
print("=" * 75)
print(f"🎉 INFERENZA COMPLETATA IN {t_total_sec:.2f} SECONDI! (~{t_total_sec / len(test_files):.2f}s per immagine)")
print(f"📊 Totale filamenti segmentati: {total_filaments_found}")
print("=" * 75)


## 📝 10. Strict Schema Validation & Submission Generation
Perform strict validation checks on generated predictions (correct column schema, non-empty masks, valid unique IDs `<image>_<n>`, and zero instance overlap) and export `/kaggle/working/submission.csv`.

In [ ]:
# ==============================================================================
# 10. STRICT SCHEMA VALIDATION & SUBMISSION FILE CREATION
# ==============================================================================

def validate_submission_df(frame, test_ids, size=2048):
    """
    Controlli bloccanti: colonne, non vuoto, ID unici e ben formati (<immagine>_<n>), immagine di
    test nota, nessun nullo, ogni RLE decodificabile 2048x2048 con almeno un pixel, e nessuna
    sovrapposizione tra istanze della stessa immagine. Restituisce un dict di statistiche.
    """
    if list(frame.columns) != ["filament_id", "segmentation_rle"]:
        raise ValueError(f"❌ Colonne errate: {list(frame.columns)}")
    if frame.empty:
        raise ValueError("❌ La submission è vuota!")
    if frame.isnull().any().any():
        raise ValueError("❌ Trovati valori nulli nel dataframe!")
    if not frame["filament_id"].is_unique:
        raise ValueError("❌ filament_id duplicati!")

    rles_by_image = defaultdict(list)
    for fid, counts in zip(frame["filament_id"], frame["segmentation_rle"]):
        stem, sep, idx = str(fid).rpartition("_")
        if not sep or not idx.isdigit():
            raise ValueError(f"❌ filament_id non valido (atteso <immagine>_<n>): {fid}")
        if stem not in test_ids:
            raise ValueError(f"❌ Immagine di test sconosciuta in filament_id: {fid}")
        rle = {"size": [size, size], "counts": str(counts).strip().encode("ascii")}
        mask = mask_utils.decode(rle)
        if mask.shape != (size, size) or not mask.any():
            raise ValueError(f"❌ RLE non valido o vuoto: {fid}")
        rles_by_image[stem].append(rle)

    for stem, rles in rles_by_image.items():
        if len(rles) > 1:
            ious = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
            if np.any(np.triu(ious, k=1) > 0):
                raise ValueError(f"❌ Istanze sovrapposte nell'immagine {stem}!")

    return {
        "rows": int(len(frame)),
        "covered_images": int(len(rles_by_image)),
        "missing_images": int(len(set(test_ids) - set(rles_by_image))),
    }


# Rimuovi file vecchio se presente
if os.path.exists(SUBMISSION_CSV_PATH):
    try:
        os.remove(SUBMISSION_CSV_PATH)
    except Exception:
        pass

# Costruzione DataFrame e validazione PRIMA del salvataggio
df_sub = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])

print("=" * 75)
print("🔍 REPORT DI VALIDAZIONE SUBMISSION:")
print("=" * 75)

test_ids = {p.stem for p in test_files}
val_stats = validate_submission_df(df_sub, test_ids)
df_sub.to_csv(SUBMISSION_CSV_PATH, index=False)

sub_p = Path(SUBMISSION_CSV_PATH)
if not sub_p.exists() or sub_p.stat().st_size == 0:
    raise RuntimeError("❌ ERRORE CRITICO: Il file submission.csv è vuoto o non creato!")
print(f"✅ File generato con successo: {sub_p} ({sub_p.stat().st_size / 1024:.2f} KB)")
print(f"✅ Schema colonne, ID, RLE 2048x2048 non vuoti e zero-overlap: OK")
print(f"✅ Immagini con almeno un filamento: {val_stats['covered_images']}/{len(test_files)}")
if val_stats["missing_images"] > 0:
    print(f"⚠️ Immagini senza alcuna predizione: {val_stats['missing_images']}")
print(f"✅ Record totali: {val_stats['rows']}")

print("\n📋 Anteprima submission.csv:")
display(df_sub.head())

print("=" * 75)
print("📥 LINK PER IL DOWNLOAD DIRETTO:")
display(FileLink(str(sub_p)))
print("=" * 75)
